# Exoplanet Habitability

Goal: use the NASA Exoplanet Archive's Planetary Systems Composite Parameters table (`pscomppars`) to find rocky exoplanets in the habitable zone, then compare our list with NASA-linked research (Kopparapu et al. 2014, the HWO target stars, and Bohl et al. 2026).

## Setup

This cell finds the project folder and adds it to Python's search path, so we can import our own tested code with `from src import physics`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The notebook usually runs from inside notebooks/, so the project root is one level up
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import physics

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "figures"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

# Create any missing folders (does nothing if they already exist)
for folder in [RAW_DIR, PROCESSED_DIR, FIGURES_DIR, OUTPUTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("pandas", pd.__version__, "| numpy", np.__version__)

### pandas 2 vs pandas 3: what changes for us

This notebook runs on pandas 2.x (what Google Colab has) and pandas 3.x (local). The setup cell above prints which version you are using. Two differences matter here:

**1. Copy-on-Write (the default in pandas 3).** A filtered DataFrame behaves like its own independent copy. So chained assignment such as
`df[df["pl_rade"] > 0]["label"] = "x"` silently does nothing in pandas 3 (pandas 2 only warns with `SettingWithCopyWarning`).
*Our rule:* always assign with `df.loc[mask, "col"] = value`, and call `.copy()` when we keep a filtered subset that we will modify later.

**2. New default string dtype.** Text columns such as `pl_name` show up as `str` in pandas 3 but `object` in pandas 2, so `df.dtypes` looks different in Colab. The data is the same.
*Our rule:* never select text columns with `select_dtypes(include="object")` (in pandas 3 it misses them). Use `pd.api.types.is_string_dtype(...)` or list the columns by name.

When a later output differs between versions, a note there points back to this cell.

## Phase 1: Load the data

### How we get the data: TAP and ADQL

**TAP** (Table Access Protocol) is a standard web interface that astronomy archives use to answer database queries. **ADQL** is the SQL-like language we send to it, for example `SELECT pl_name, pl_rade FROM pscomppars`. We send the query as a web request with `requests` and the archive sends back a CSV file.

### Why the Composite Parameters table (`pscomppars`)?

The default Planetary Systems table (`ps`) keeps **every published solution** for every planet, so one planet can appear many times. When we checked, `ps` had 40,350 rows, and TRAPPIST-1 e alone had 5 rows. You would have to pick one solution per planet yourself. `pscomppars` has **one row per planet**, filled with the best available value for each column, which is what a planet-by-planet analysis needs.

**Caveat from the archive:** the values in one row can come from different papers, so a row is *"not necessarily self-consistent"*. Some values are also **calculated** by the archive when a measurement is missing. For example, it estimates a radius from the mass using the Chen & Kipping (2017) relation. That's why we also keep three source columns: `pl_rade_reflink`, `pl_bmassprov` and `pl_insol_reflink`. ([column docs](https://exoplanetarchive.ipac.caltech.edu/docs/API_PS_columns.html), [calculated values](https://exoplanetarchive.ipac.caltech.edu/docs/pscp_calc.html))

In [ ]:
from datetime import date

from src import archive

COLUMNS = [
    "pl_name", "hostname", "discoverymethod", "disc_year",
    "pl_rade", "pl_bmasse", "pl_orbper", "pl_orbsmax", "pl_orbeccen",
    "pl_insol", "pl_eqt",
    "st_teff", "st_rad", "st_lum", "st_mass", "sy_dist",
    # Source columns: say whether a value was measured or calculated by the archive
    "pl_rade_reflink", "pl_bmassprov", "pl_insol_reflink",
]

adql = "SELECT " + ", ".join(COLUMNS) + " FROM pscomppars"
print(adql)

**Load or download.** If a snapshot already exists in `data/raw/`, we load the newest one and do **not** call the API. Set `REFRESH = True` to take a new dated snapshot; old snapshots are never overwritten.

In [ ]:
REFRESH = False  # set to True to take a new dated snapshot

todays_file = RAW_DIR / f"pscomppars_{date.today().isoformat()}.csv"
snapshot = archive.latest_snapshot(RAW_DIR, "pscomppars")

if snapshot is None or (REFRESH and not todays_file.exists()):
    print("Downloading a new snapshot from the archive...")
    snapshot = archive.download_tap_csv(adql, todays_file)
else:
    print("Loading existing snapshot (no API call)")

print("File:", snapshot.relative_to(PROJECT_ROOT).as_posix())
planets = pd.read_csv(snapshot)

# If the archive returned an error message instead of data, the header would not match
assert list(planets.columns) == COLUMNS, f"Unexpected columns: {list(planets.columns)}"

print("Shape (rows, columns):", planets.shape)
print(f"Planets in table: {len(planets):,}")

**Data types.** Numbers are `float64` or `int64`. Text columns show as `str` on pandas 3 and `object` on pandas 2; the data is the same (see the *pandas 2 vs pandas 3* cell).

In [ ]:
planets.dtypes.to_frame("dtype")

**Summary statistics.** `describe()` gives the count, mean, spread and percentiles of each numeric column. We transpose it (`.T`) so each column becomes a row, which is easier to read.

In [ ]:
planets.describe().T

**Percent missing per column.** For each column, the share of planets with no value. This tells us which later steps will lose planets.

In [ ]:
percent_missing = (
    planets.isna().mean()   # fraction of rows that are missing, per column
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
    .to_frame("percent_missing")
)
percent_missing

**Measured or calculated?** The reference column says "Calculated Value" when the archive computed the number itself instead of taking it from a paper. `pl_bmassprov` tells us whether the mass is a true mass, a minimum mass (`Msini`, from radial velocity), or estimated from the radius (`M-R relationship`).

In [ ]:
radius_calculated = planets["pl_rade_reflink"].str.contains("Calculated Value", na=False)
insol_calculated = planets["pl_insol_reflink"].str.contains("Calculated Value", na=False)

print(f"Radii calculated from mass by the archive: {radius_calculated.sum():,} "
      f"of {planets['pl_rade'].notna().sum():,} planets with a radius")
print(f"Insolations calculated by the archive:     {insol_calculated.sum():,} "
      f"of {planets['pl_insol'].notna().sum():,} planets with an insolation")
print()
print("Where does the mass come from? (pl_bmassprov)")
print(planets["pl_bmassprov"].value_counts(dropna=False).to_string())

In [ ]:
# Calculated radii by discovery method: radial velocity planets almost never have a measured radius
pd.crosstab(planets["discoverymethod"], radius_calculated.rename("radius_calculated"))

### What stands out (snapshot of 2026-10-24)

- **6,451 planets**, one row each. Radius is missing for only 0.8%, but eccentricity is missing for 16.6% and insolation for 9.2%. Dropping every row with *any* missing value would throw away far more planets than any single step needs (Phase 2).
- **Many radii are not measured.** 1,539 of 6,401 radii were calculated by the archive from the mass. That covers 1,134 of the radial velocity planets and all 292 microlensing planets, but only 2 transit planets. A "rocky" label based on these radii is really a mass-based guess.
- **Masses are often estimates too.** 3,035 masses come from the mass-radius relationship (mostly transit planets with no measured mass), and 924 are only minimum masses (`Msini`).
- **Odd values to handle in Phase 2.** 46 planets have `pl_insol` = 0. These are wide-orbit imaged planets whose tiny flux rounds to zero, not true zeros; we'll recompute S in Phase 3. 3,156 of 5,383 eccentricities are exactly 0, which usually means "assumed circular", not "measured circular". The hottest host (57,000 K) is a white dwarf in the NN Ser system.
- **Huge ranges** (orbital period from 0.09 days to 400 million days, insolation from 0 to 636,000 times Earth's). That's why we will use log scales for plots and clustering.

## Phase 2: Clean and filter

### Why we don't drop every row with a missing value

`planets.dropna()` would delete a planet if *any* column is empty. For example, it would delete a planet just because its eccentricity is unknown (16.6% of planets), even though deciding whether a planet is rocky doesn't need eccentricity. That throws away good data and can bias the sample toward well-studied systems. Instead, each step only drops planets missing the columns *that step* needs. The tradeoff is that different steps use slightly different sets of planets, so we print how many planets remain after every step.

### Step 1: remove physically impossible values

A radius, mass, orbit, temperature or distance of zero or less can't be real. We replace those values with `NaN` (missing) instead of deleting the planet, so it stays available for steps that don't need that column. Eccentricity must satisfy 0 ≤ e < 1 for a closed orbit. `st_lum` is not checked: it is a log10, so negative values are normal. The 46 zero insolations are wide-orbit planets whose tiny flux rounds to 0; Phase 3 recomputes them.

In [ ]:
from src import rocky

assert planets["pl_name"].is_unique, "pl_name should identify each planet once"

clean, removed = rocky.clean_impossible_values(planets)
print(removed[removed > 0].to_string())
print()
print(f"Planets after cleaning: {len(clean):,} (bad values removed, no rows dropped)")

### Step 2: which evidence do we use for each planet?

**Minimum mass.** The radial velocity method measures how much a star wobbles toward and away from us as a planet pulls on it. If the orbit is tilted, we only see part of the wobble, so we get M·sin(i): a *lower bound* on the true mass, never the mass itself.

Each planet gets a `rocky_path`:
1. **excluded**: microlensing planets. Their `pl_orbsmax` is a projected separation, and 290 of 292 have no insolation, so we can't place them in a habitable zone.
2. **measured radius**: a real radius measurement (`pl_rade_reflink` is not "Calculated Value"). This includes 33 planets found by radial velocity that were later seen transiting.
3. **mass only**: no measured radius, but a measured mass (a minimum mass or a true mass). We ignore the archive's estimated radius. `mass_type` records whether the mass is a minimum (`Msini`) or a true mass (`Mass`, `Msin(i)/sin(i)`).
4. **excluded**: no measured radius and no usable mass.

In [ ]:
classified = rocky.assign_rocky_path(clean)

not_microlensing = classified["discoverymethod"] != "Microlensing"
print(f"Planets before path assignment:     {len(classified):,}")
print(f"After excluding microlensing:       {not_microlensing.sum():,}")
usable = classified["rocky_path"] != rocky.PATH_EXCLUDED
print(f"After excluding no usable data:     {usable.sum():,}")
print()
print(classified["rocky_path"].value_counts().to_string())

In [ ]:
# Which discovery methods end up on each path?
pd.crosstab(classified["discoverymethod"], classified["rocky_path"], margins=True)

In [ ]:
on_mass_path = classified["rocky_path"] == rocky.PATH_MASS_ONLY
print("Mass type on the mass only path:")
print(classified.loc[on_mass_path, "mass_type"].value_counts().to_string())

### Step 3: rocky labels

**The radius valley.** Very few planets have radii between about 1.5 and 2.0 Earth radii (Fulton et al. 2017, AJ 154, 109). The usual explanation: planets just below the gap lost their gas envelopes to their star's radiation and are bare rock, while planets above it kept a puffy hydrogen layer ("mini-Neptunes"). 1.6 R⊕ is a common rocky cutoff because Rogers (2015, arXiv:1407.4457) showed that most planets of that size are already too low-density to be pure rock and iron.

| Path | rocky | possibly rocky | not rocky |
|---|---|---|---|
| measured radius | R ≤ 1.6 R⊕ | 1.6 < R ≤ 2.0 R⊕ | R > 2.0 R⊕ |
| mass only | (none) | M ≤ 5 M⊕ | M > 5 M⊕ |

**Judgment calls (value, source, alternative):**
- **1.6 R⊕**: Rogers (2015). Alternative: 1.5 R⊕, the lower edge of the Fulton valley.
- **2.0 R⊕**: Bohl et al. (2026, arXiv:2501.14054), §1: *"we adopt a maximum radius of 2 R⊕"*; §3: *"≤2 R⊕, or ≤5 M⊕ for non-transiting planets"* (both cite Hill et al. 2023). So our "rocky + possibly rocky" matches their rocky set, which keeps the Phase 6 comparison fair. Alternative: 1.96 R⊕, the top of the range Bohl et al. quote.
- **5 M⊕**: Bohl et al. §1: *"For planets without measured radii, we adopt a maximum mass of 5 M⊕"*. Alternative: about 6 M⊕, the Earth-like-composition mass at 1.6 R⊕.
- **Mass-path planets are at most "possibly rocky"**: a minimum mass is only a lower bound, and mass alone doesn't tell us what a planet is made of. Alternative: call M ≤ 5 "rocky", as Bohl et al. do.

In [ ]:
classified = rocky.label_rockiness(classified)

has_label = classified["rocky_label"] != ""
print(pd.crosstab(classified.loc[has_label, "rocky_path"],
                  classified.loc[has_label, "rocky_label"], margins=True).to_string())
print()
small = classified["rocky_label"].isin(["rocky", "possibly rocky"])
print(f"Planets with a rocky label:            {has_label.sum():,}")
print(f"Planets that are rocky or possibly rocky: {small.sum():,}")

### Why we never use a mass computed from the radius (circularity)

For 3,035 planets, the archive did not measure the mass. It *computed* the mass from the radius using the Chen & Kipping (2017) formula. If we then asked "is the mass ≤ 5 M⊕?" to confirm that a planet is rocky, we would just be checking the radius again with extra steps: the formula turns R = 2.0 R⊕ into M ≈ 4.7 M⊕, so the two tests agree **by construction**. Agreement like that is not evidence. The table below shows it: for planets with computed masses the agreement is near 100% and means nothing. For planets with a real measured mass, any disagreement is real information about density (a puffy planet or a dense one).

**Rule used everywhere in this project:** a mass with `pl_bmassprov == "M-R relationship"` is never used to support, confirm, or change a rocky label. Our classifier only uses mass for planets *without* a measured radius, and tests in `tests/test_rocky.py` enforce this.

In [ ]:
radius_path = classified[classified["rocky_path"] == rocky.PATH_MEASURED_RADIUS]
radius_says_small = radius_path["pl_rade"] <= rocky.POSSIBLY_ROCKY_MAX_RADIUS
mass_says_small = radius_path["pl_bmasse"] <= rocky.ROCKY_MAX_MASS

for provenance in ["M-R relationship", "Mass"]:
    group = (radius_path["pl_bmassprov"] == provenance) & radius_path["pl_bmasse"].notna()
    agree = (radius_says_small[group] == mass_says_small[group]).mean() * 100
    print(f"Mass from '{provenance}': {group.sum():,} planets, "
          f"radius and mass tests agree {agree:.1f}% of the time")

**Why only 97%, not 100%, for computed masses?** The two cutoffs are not exactly equivalent under the formula: 2.0 R⊕ becomes 4.66 M⊕, and 5 M⊕ becomes about 2.09 R⊕. The next cell shows that every disagreement is a planet between 2.00 and 2.09 R⊕, the sliver between the two cutoffs. It also recomputes the masses from the radii with the Chen & Kipping (2017) formula, using the coefficients the archive lists ([pscp_calc](https://exoplanetarchive.ipac.caltech.edu/docs/pscp_calc.html): log10 R = −0.0925 + 0.589 · log10 M for 2.04–132 M⊕). The archive's "masses" match the formula almost exactly, so they carry no information the radius doesn't already have.

In [ ]:
computed = radius_path[(radius_path["pl_bmassprov"] == "M-R relationship")
                       & radius_path["pl_bmasse"].notna()]
disagree = (computed["pl_rade"] <= rocky.POSSIBLY_ROCKY_MAX_RADIUS) != (computed["pl_bmasse"] <= rocky.ROCKY_MAX_MASS)
print(f"Disagreements: {disagree.sum()}, all with radius between "
      f"{computed.loc[disagree, 'pl_rade'].min():.2f} and {computed.loc[disagree, 'pl_rade'].max():.2f} R⊕")

# Invert log10(R) = -0.0925 + 0.589 * log10(M) to predict M from R (valid for about 1.23-11.1 R⊕)
middle = computed[computed["pl_rade"].between(1.23, 11.1)]
predicted_mass = 10 ** ((np.log10(middle["pl_rade"]) + 0.0925) / 0.589)
ratio = middle["pl_bmasse"] / predicted_mass
print(f"Archive mass / formula mass for {len(middle):,} planets: "
      f"min {ratio.min():.3f}, median {ratio.median():.3f}, max {ratio.max():.3f}")

In [ ]:
processed_file = PROCESSED_DIR / "planets_classified.csv"
classified.to_csv(processed_file, index=False)
print("Saved", processed_file.relative_to(PROJECT_ROOT).as_posix(), classified.shape)

### Step 4: see the radius valley ourselves

We plot measured radii on a **log scale**: equal steps on the axis mean equal *ratios* (1 → 2 is as wide as 10 → 20), which spreads out the small planets where the valley is. The left panel shows all radii, with the archive's calculated radii drawn as an outline so we can see how estimated values behave. The right panel zooms in on planets with orbital periods under 100 days, the same cut Fulton et al. (2017) used, with finer bins.

**Why do the estimated radii spike at 12–15 R⊕?** The archive's Chen & Kipping (2017) relation is almost flat between 132 and 26,600 M⊕ (R ∝ M^−0.044), so every radial velocity giant from about Saturn mass up to brown dwarfs gets a radius of 12–14 R⊕, and they all pile into one spike.

In [ ]:
from src import plot_style

plot_style.apply_style()

on_radius_path = classified["rocky_path"] == rocky.PATH_MEASURED_RADIUS
measured_radii = classified.loc[on_radius_path, "pl_rade"]
calculated_radii = classified.loc[rocky.radius_is_calculated(classified), "pl_rade"]
short_period_radii = classified.loc[on_radius_path & (classified["pl_orbper"] < 100), "pl_rade"]

fig, (ax_all, ax_zoom) = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [3, 2]})


def draw_cutoffs(ax):
    """Shade Fulton's valley and draw dashed lines at our two radius cutoffs."""
    ax.axvspan(1.5, 2.0, color=plot_style.SHADE, zorder=0,
               label="Radius valley, 1.5–2.0 R⊕ (Fulton et al. 2017)")
    for cutoff in [rocky.ROCKY_MAX_RADIUS, rocky.POSSIBLY_ROCKY_MAX_RADIUS]:
        ax.axvline(cutoff, color=plot_style.TEXT_SECONDARY, linestyle="--", linewidth=1)


# Left: all radii. Log-spaced bins, each about 9% wider than the one before
bins_all = np.logspace(np.log10(0.3), np.log10(40), 57)
draw_cutoffs(ax_all)
ax_all.hist(measured_radii, bins=bins_all, color=plot_style.SERIES[0], edgecolor=plot_style.SURFACE,
            linewidth=0.8, label=f"Measured radius ({len(measured_radii):,} planets)")
ax_all.hist(calculated_radii, bins=bins_all, histtype="step", color=plot_style.SERIES[1], linewidth=1.6,
            label=f"Calculated from mass by the archive ({len(calculated_radii):,})")
ax_all.set_xscale("log")
ax_all.set_xlim(0.3, 40)
ax_all.set_xticks([0.5, 1, 1.6, 2, 3, 5, 10, 20])
ax_all.set_xticklabels(["0.5", "1", "1.6", "2", "3", "5", "10", "20"])
ax_all.set_xlabel("Planet radius (Earth radii, log scale)")
ax_all.set_ylabel("Number of planets")
ax_all.set_title("All planets with a radius")
ax_all.set_ylim(0, ax_all.get_ylim()[1] * 1.45)  # headroom so the legend sits above the bars
ax_all.legend(loc="upper right")

# Right: zoom on small, short-period planets with finer bins
bins_zoom = np.logspace(np.log10(0.7), np.log10(4.5), 31)
draw_cutoffs(ax_zoom)
ax_zoom.hist(short_period_radii, bins=bins_zoom, color=plot_style.SERIES[0],
             edgecolor=plot_style.SURFACE, linewidth=0.8,
             label=f"Measured radius, period < 100 days ({short_period_radii.between(0.7, 4.5).sum():,} shown)")
ax_zoom.set_xscale("log")
ax_zoom.set_xlim(0.7, 4.5)
ax_zoom.set_xticks([0.8, 1, 1.2, 1.6, 2, 2.5, 3, 4])
ax_zoom.set_xticklabels(["0.8", "1", "1.2", "1.6", "2", "2.5", "3", "4"])
ax_zoom.set_xlabel("Planet radius (Earth radii, log scale)")
ax_zoom.set_ylabel("Number of planets")
ax_zoom.set_title("Zoom: small planets, orbital period < 100 days")
ax_zoom.set_ylim(0, ax_zoom.get_ylim()[1] * 1.3)
ax_zoom.legend(loc="upper left")

fig.suptitle("Exoplanet radii and the radius valley (dashed lines: our 1.6 and 2.0 R⊕ cutoffs)",
             fontsize=14, color=plot_style.TEXT_PRIMARY)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "radius_histogram.png")
plt.show()

### What we learned in Phase 2

- **Cleaning removed values, not planets.** Only `pl_insol` had impossible values: 46 zeros, which are rounding for wide orbits. All 6,451 planets stay in the table.
- **Paths:** 4,862 planets use a measured radius, 1,290 use a mass ("mass only" path: 877 minimum masses, 413 true masses), and 299 are excluded (292 microlensing, 7 with no usable radius or mass).
- **Labels:** 1,143 rocky, 665 possibly rocky (567 by radius, 98 by mass), 4,344 not rocky, so **1,808 planets are rocky or possibly rocky** and move on to the habitable zone step.
- **Circularity is real, not hypothetical.** The archive's radius-derived masses match the Chen & Kipping formula to within 0.5%, so their 97% "agreement" with the radius is guaranteed. Real measured masses agree only 90% of the time, and that 10% is real information about density.
- **The radius valley is faint in this data.** We see a dip near 1.8–1.9 R⊕, not a clean gap, because the archive mixes radii of very different precision. Estimated radii also create a fake spike at 12–15 R⊕, another reason to keep them out of the analysis.

## Phase 3: Stellar energy

### Step 0: when could Bohl et al. have seen each planet?

Bohl et al. downloaded the archive on **December 20, 2025** (their §2). Planets added or updated after that date could not be in their catalog, so Phase 6 sets them aside. `pscomppars` has no release or update date (only `disc_pubdate`, the discovery *publication* month), so we use the `ps` table, which keeps a `releasedate` ("Date the parameter set was publicly released") and a `rowupdate` for every published solution:
- `new_since_bohl`: the planet's **earliest** `releasedate` is after 2025-12-20.
- `updated_since_bohl`: the planet's **latest** `rowupdate` is after 2025-12-20 (so its values may have changed).
- `after_bohl_cutoff`: either one.

We also download the eccentricity uncertainty and limit-flag columns, which we need in Step 3. Both are new dated snapshots; the main Phase 1 snapshot is not touched.

In [ ]:
PS_DATES_QUERY = ("SELECT pl_name, MIN(releasedate) AS first_release, MAX(releasedate) AS last_release, "
                  "MAX(rowupdate) AS last_rowupdate FROM ps GROUP BY pl_name")
EXTRA_QUERY = ("SELECT pl_name, disc_pubdate, pl_orbeccen, pl_orbeccenlim, pl_orbeccenerr1, pl_orbeccenerr2 "
               "FROM pscomppars")


def load_or_download(prefix, adql):
    """Load the newest dated snapshot with this prefix, downloading one only if none exists."""
    snapshot_file = archive.latest_snapshot(RAW_DIR, prefix)
    if snapshot_file is None:
        print(f"Downloading {prefix} ...")
        snapshot_file = archive.download_tap_csv(adql, RAW_DIR / f"{prefix}_{date.today().isoformat()}.csv")
    print("Loaded", snapshot_file.relative_to(PROJECT_ROOT).as_posix())
    return pd.read_csv(snapshot_file)


ps_dates = load_or_download("ps_dates", PS_DATES_QUERY)
extra = load_or_download("pscomppars_extra", EXTRA_QUERY)

# Same planets as our main snapshot, and identical eccentricities (so the downloads are consistent)
assert set(extra["pl_name"]) == set(classified["pl_name"])
assert set(classified["pl_name"]) <= set(ps_dates["pl_name"])
check = classified[["pl_name", "pl_orbeccen"]].merge(extra[["pl_name", "pl_orbeccen"]], on="pl_name",
                                                     suffixes=("", "_extra"))
assert ((check["pl_orbeccen"] == check["pl_orbeccen_extra"]) | check["pl_orbeccen_extra"].isna()
        | (check["pl_orbeccen_extra"] >= 1)).all()

energy = classified.merge(ps_dates, on="pl_name", how="left").merge(
    extra.drop(columns="pl_orbeccen"), on="pl_name", how="left")

BOHL_DOWNLOAD_DATE = "2025-12-20"  # Bohl et al. (2026) Section 2
energy["new_since_bohl"] = energy["first_release"] > BOHL_DOWNLOAD_DATE
energy["updated_since_bohl"] = energy["last_rowupdate"] > BOHL_DOWNLOAD_DATE
energy["after_bohl_cutoff"] = energy["new_since_bohl"] | energy["updated_since_bohl"]

print()
print(f"New since Bohl et al. (first released after {BOHL_DOWNLOAD_DATE}): {energy['new_since_bohl'].sum():,}")
print(f"Updated since Bohl et al. (last row update after it):            {energy['updated_since_bohl'].sum():,}")
print(f"Either (after_bohl_cutoff):                                       {energy['after_bohl_cutoff'].sum():,}")
print(f"Cross-check: discovery publication month 2025-12 or later:       {(energy['disc_pubdate'] >= '2025-12').sum():,}")

### Step 1: how much energy does each planet receive?

**Luminosity** is the total power a star gives off. The archive stores it as log10(L / L☉), so we undo the log with 10^x. When it is missing we estimate it with the **Stefan–Boltzmann law**: a star's power grows with its surface area (R²) and very steeply with temperature (T⁴), so L = R² (Teff / 5772 K)⁴ in solar units (5772 K is the Sun's nominal temperature, IAU 2015 Resolution B3).

**Insolation** S is the flux reaching the planet in Earth units. Light spreads over a sphere whose area grows as distance squared, so S = L / a² (the inverse-square law): a planet twice as far gets one quarter of the energy.

In [ ]:
from src import physics

energy["luminosity"] = physics.luminosity_from_log(energy["st_lum"])
estimated_lum = physics.luminosity_from_radius_teff(energy["st_rad"], energy["st_teff"])
fill_lum = energy["luminosity"].isna() & estimated_lum.notna()
energy.loc[fill_lum, "luminosity"] = estimated_lum[fill_lum]
energy["luminosity_source"] = np.where(fill_lum, "from R and Teff",
                                       np.where(energy["luminosity"].notna(), "archive st_lum", "missing"))
print(energy["luminosity_source"].value_counts().to_string())
print(f"Filled from R and Teff: {fill_lum.sum()} (the archive already computes L from R and Teff when it can)")

energy["insol_ours"] = physics.insolation(energy["luminosity"], energy["pl_orbsmax"])
print()
print(f"Planets with our computed S: {energy['insol_ours'].notna().sum():,}")

**Do our values agree with the archive's `pl_insol`?** We compare them on a log–log scatter plot. Points on the dashed 1:1 line agree exactly. We split the points by whether the archive's own value was *calculated* (from the same L / a² formula, so they should agree almost perfectly) or taken from a paper (which may use different stellar parameters).

In [ ]:
from src import plot_style

plot_style.apply_style()

both = energy["insol_ours"].notna() & energy["pl_insol"].notna()
archive_calculated = energy["pl_insol_reflink"].str.contains("Calculated Value", na=False)
percent_diff = (energy["insol_ours"] - energy["pl_insol"]) / energy["pl_insol"] * 100

print(f"Planets with both values: {both.sum():,}")
print(f"Median percent difference, all:                 {percent_diff[both].median():+.2f}%")
print(f"  archive value calculated by the archive ({(both & archive_calculated).sum():,}): "
      f"{percent_diff[both & archive_calculated].median():+.2f}%")
print(f"  archive value from a paper ({(both & ~archive_calculated).sum():,}):              "
      f"{percent_diff[both & ~archive_calculated].median():+.2f}%")
print(f"Median absolute percent difference (from a paper): {percent_diff[both & ~archive_calculated].abs().median():.1f}%")

fig, ax = plt.subplots(figsize=(7.5, 7))
groups = [(~archive_calculated, "Archive value from a paper", plot_style.SERIES[0]),
          (archive_calculated, "Archive value calculated by the archive", plot_style.SERIES[1])]
for mask, label, color in groups:
    subset = energy[both & mask]
    ax.scatter(subset["pl_insol"], subset["insol_ours"], s=10, alpha=0.55, color=color,
               edgecolors="none", label=f"{label} ({len(subset):,})")
limits = [1e-4, 1e6]
ax.plot(limits, limits, linestyle="--", color=plot_style.TEXT_SECONDARY, linewidth=1, label="1:1 line (perfect agreement)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(limits)
ax.set_ylim(limits)
ax.set_xlabel("Archive insolation pl_insol (Earth flux, log scale)")
ax.set_ylabel("Our insolation L / a² (Earth flux, log scale)")
ax.set_title("Our computed insolation vs the archive's")
ax.legend(loc="upper left")
fig.savefig(FIGURES_DIR / "insolation_comparison.png")
plt.show()

In [ ]:
# Use the archive's value where it exists, and ours to fill the gaps (project brief)
energy["insol"] = energy["pl_insol"].fillna(energy["insol_ours"])
filled = energy["pl_insol"].isna() & energy["insol_ours"].notna()
print(f"Archive insolation available: {energy['pl_insol'].notna().sum():,}")
print(f"Gaps filled with our value:   {filled.sum():,}")
print(f"Planets with an insolation:   {energy['insol'].notna().sum():,} of {len(energy):,}")

### Step 2: why eccentric orbits receive more energy on average

On an eccentric (oval) orbit, the planet's distance changes, and flux goes as 1/r². Close to the star the flux shoots up a lot; far away it drops by less than that. The close passes win, so averaged over time the planet gets **more** energy than a circular orbit at the same semimajor axis: ⟨S⟩ = S / √(1 − e²) (Williams & Pollard 2002; Bolmont et al. 2016, the paper Bohl et al. follow). For e = 0.3 that is about 5% more; for e = 0.6 it is 25% more.

We only apply this where the eccentricity was **measured**: limit flag 0 (a measurement, not an upper limit) **and** a reported uncertainty. An e of exactly 0 with no uncertainty usually means the authors *assumed* a circular orbit. For those, and for upper limits, `flux_orbit_avg` stays empty and we use the plain S, the same as Bohl et al., who set upper limits to e = 0 (§2).

In [ ]:
e = energy["pl_orbeccen"]
has_uncertainty = energy["pl_orbeccenerr1"].notna() | energy["pl_orbeccenerr2"].notna()
is_measured = (energy["pl_orbeccenlim"] == 0) & has_uncertainty & (e >= 0) & (e < 1)

energy["ecc_status"] = np.select(
    [e.isna(), is_measured, energy["pl_orbeccenlim"] == 1, (e == 0) & ~has_uncertainty],
    ["missing", "measured", "upper limit", "assumed zero"],
    default="no uncertainty or other flag",
)
print(energy["ecc_status"].value_counts().to_string())

energy["flux_orbit_avg"] = np.where(is_measured, physics.orbit_averaged_flux(energy["insol"], e), np.nan)
energy["hz_flux"] = energy["flux_orbit_avg"].fillna(energy["insol"])

boost = (energy["flux_orbit_avg"] / energy["insol"] - 1) * 100
print()
print(f"Planets with an orbit-averaged flux: {energy['flux_orbit_avg'].notna().sum():,}")
print(f"Median boost {boost.median():.2f}%, more than 5% for {(boost > 5).sum():,} planets, "
      f"more than 25% for {(boost > 25).sum():,}")

### Step 3: equilibrium temperature

The **equilibrium temperature** is the temperature a planet would settle at if it were a simple ball that absorbs starlight and re-radiates it as heat, with no greenhouse effect. The **albedo** is the fraction of light reflected straight back to space (Earth ≈ 0.3). T_eq = T₀ · S^¼ · (1 − albedo)^¼.

**Judgment call (T₀):** the brief suggested 278.6 K, which comes from an older solar constant (1367 W m⁻²). We compute T₀ = (S₀ / 4σ)^¼ from the IAU 2015 nominal S₀ = 1361 W m⁻² and the exact Stefan–Boltzmann constant σ, which gives 278.3 K. Alternative: 278.6 K. Earth comes out at about 255 K either way, about 33 K colder than its real average of 288 K. That gap is Earth's greenhouse effect.

The archive's `pl_eqt` assumes **no** albedo (its calculation page), so to compare like with like we also compute our albedo-0 version from the plain insolation.

In [ ]:
print(f"T0 = {physics.T_EQ_EARTH_NO_ALBEDO:.1f} K; Earth at albedo 0.3: {physics.equilibrium_temperature(1.0, 0.3):.1f} K")

energy["teq_albedo_0.3"] = physics.equilibrium_temperature(energy["hz_flux"], 0.3)
teq_albedo_0 = physics.equilibrium_temperature(energy["insol"], 0.0)

both_teq = teq_albedo_0.notna() & energy["pl_eqt"].notna()
teq_diff = (teq_albedo_0 - energy["pl_eqt"]) / energy["pl_eqt"] * 100
print(f"Our albedo-0 T_eq vs archive pl_eqt ({both_teq.sum():,} planets): "
      f"median difference {teq_diff[both_teq].median():+.2f}%")

### Step 4: habitable zone flags

The **habitable zone** is the range of flux where a rocky planet with an Earth-like atmosphere could keep liquid water on its surface. Too much flux and the oceans boil off (inner edge); too little and even a thick CO₂ atmosphere can't keep it warm (outer edge). The edges depend on the star's temperature, because red light from cool stars is absorbed more easily and reflected less.

| Column | Inner edge / outer edge | Source | Temperature range |
|---|---|---|---|
| `hz_simple_conservative` | 1.107 / 0.356 S⊕ (the Sun's values) | Kopparapu et al. 2014, Table 1 | ignores Teff |
| `hz_simple_optimistic` | 1.776 / 0.32 S⊕ (the Sun's values) | Kopparapu et al. 2014, Table 1 | ignores Teff |
| `hz_k14_conservative` | Runaway Greenhouse (1 M⊕) / Maximum Greenhouse | Kopparapu et al. 2014, Table 1, Eq. 4 | 2600–7200 K (§3.1) |
| `hz_k14_optimistic` | Recent Venus / Early Mars | Kopparapu et al. 2014, Table 1, Eq. 4 | 2600–7200 K |
| `hz_bohl_empirical` | Recent Venus / Early Mars | Kopparapu et al. 2013, Table 3 (cited by Bohl §1) | 2600–7200 K (§4.1) |
| `hz_bohl_3d` | Leconte et al. 2013 3D limit / Early Mars | Ramirez & Kaltenegger 2018, Table 2 (cited by Bohl §1) | 2600–7200 K |

**Outside the temperature range we don't extrapolate**: those planets are labelled "outside model range" (your rule). The runaway greenhouse edge uses the 1 M⊕ coefficients for every planet (judgment call: Bohl §2 also ignores mass, and many masses are estimates; alternative: 5 M⊕ coefficients for measured super-Earth masses). All flags use `hz_flux` (orbit-averaged where measured).

In [ ]:
from src import habitable_zone as hz

teff = energy["st_teff"]
K14, K13 = hz.KOPPARAPU_2014, hz.KOPPARAPU_2013
K_RANGE = hz.KOPPARAPU_TEFF_RANGE

energy["k14_recent_venus"] = hz.hz_flux_limit(teff, K14["recent_venus"], K_RANGE)
energy["k14_runaway_greenhouse"] = hz.hz_flux_limit(teff, K14["runaway_greenhouse"], K_RANGE)
energy["k14_maximum_greenhouse"] = hz.hz_flux_limit(teff, K14["maximum_greenhouse"], K_RANGE)
energy["k14_early_mars"] = hz.hz_flux_limit(teff, K14["early_mars"], K_RANGE)
energy["bohl_recent_venus"] = hz.hz_flux_limit(teff, K13["recent_venus"], K_RANGE)
energy["bohl_early_mars"] = hz.hz_flux_limit(teff, K13["early_mars"], K_RANGE)
energy["bohl_3d_inner"] = hz.hz_flux_limit(teff, hz.RAMIREZ_KALTENEGGER_2018_LECONTE, K_RANGE)

flux = energy["hz_flux"]
sun = K14
energy["hz_simple_conservative"] = hz.hz_status(flux, sun["runaway_greenhouse"][0], sun["maximum_greenhouse"][0])
energy["hz_simple_optimistic"] = hz.hz_status(flux, sun["recent_venus"][0], sun["early_mars"][0])
energy["hz_k14_conservative"] = hz.hz_status(flux, energy["k14_runaway_greenhouse"],
                                             energy["k14_maximum_greenhouse"], teff, K_RANGE)
energy["hz_k14_optimistic"] = hz.hz_status(flux, energy["k14_recent_venus"], energy["k14_early_mars"], teff, K_RANGE)
energy["hz_bohl_empirical"] = hz.hz_status(flux, energy["bohl_recent_venus"], energy["bohl_early_mars"], teff, K_RANGE)
energy["hz_bohl_3d"] = hz.hz_status(flux, energy["bohl_3d_inner"], energy["bohl_early_mars"], teff, K_RANGE)

HZ_COLUMNS = ["hz_simple_conservative", "hz_simple_optimistic", "hz_k14_conservative",
              "hz_k14_optimistic", "hz_bohl_empirical", "hz_bohl_3d"]
small = energy["rocky_label"].isin(["rocky", "possibly rocky"])
print(f"Rocky or possibly rocky planets: {small.sum():,}")
print(f"  ... with a flux:                {(small & flux.notna()).sum():,}")
print(f"  ... and a star temperature in 2600-7200 K: "
      f"{(small & flux.notna() & hz.in_teff_range(teff, K_RANGE)).sum():,}")
print()
summary = pd.DataFrame({column: energy.loc[small, column].value_counts() for column in HZ_COLUMNS}).fillna(0).astype(int)
summary

In [ ]:
# What did the strict temperature range remove? Planets inside the simple (Teff-free) optimistic window
# whose star is outside 2600-7200 K
removed_by_range = small & (energy["hz_simple_optimistic"] == "inside") & (energy["hz_k14_optimistic"] == "outside model range")
print(f"Small planets in the simple optimistic window but outside the model range: {removed_by_range.sum()}")
energy.loc[removed_by_range, ["pl_name", "st_teff", "hz_flux", "rocky_label"]]

**Which flux do the HZ flags use?** All flags use `hz_flux`: the orbit-averaged flux where the eccentricity was measured, otherwise the plain (circular-orbit) flux. Does that choice matter? The next cell recomputes every zone with the circular flux for all planets and counts how many small planets change.

In [ ]:
circular = energy["insol"]
zone_limits = {
    "hz_simple_conservative": (sun["runaway_greenhouse"][0], sun["maximum_greenhouse"][0], None),
    "hz_simple_optimistic": (sun["recent_venus"][0], sun["early_mars"][0], None),
    "hz_k14_conservative": (energy["k14_runaway_greenhouse"], energy["k14_maximum_greenhouse"], K_RANGE),
    "hz_k14_optimistic": (energy["k14_recent_venus"], energy["k14_early_mars"], K_RANGE),
    "hz_bohl_empirical": (energy["bohl_recent_venus"], energy["bohl_early_mars"], K_RANGE),
    "hz_bohl_3d": (energy["bohl_3d_inner"], energy["bohl_early_mars"], K_RANGE),
}
rows = []
for column, (inner, outer, teff_range) in zone_limits.items():
    with_circular = hz.hz_status(circular, inner, outer, teff, teff_range)
    inside_avg = small & (energy[column] == hz.INSIDE)
    inside_circ = small & (with_circular == hz.INSIDE)
    rows.append({"zone": column, "inside (orbit-averaged where measured)": inside_avg.sum(),
                 "inside (circular for all)": inside_circ.sum(),
                 "planets that change": (inside_avg != inside_circ).sum()})
print(f"Small planets with a measured-eccentricity (orbit-averaged) flux: "
      f"{(small & energy['flux_orbit_avg'].notna()).sum():,}")
pd.DataFrame(rows)

**Checking our limits against Bohl et al.'s full Table 1.** Bohl et al. publish their catalog on Zenodo ([10.5281/zenodo.18134528](https://doi.org/10.5281/zenodo.18134528), CC-BY-4.0). We save it as a dated raw snapshot and compare their printed limits with ours for every planet whose star is inside 2600–7200 K. Their Recent Venus and Early Mars limits should match our Kopparapu (2013) formula. Their 3D limit does not match the Ramirez & Kaltenegger (2018) polynomial their §1 names. Their CSV's own column notes cite *"Leconte et al. 2013; Ramirez & Kaltenegger 2014"* for this limit, so they likely used a different parametrization than the one we could find.

In [ ]:
BOHL_URL = "https://zenodo.org/api/records/18134528/files/Table1_rocky_HZ_Gaia_NEA.csv/content"
BOHL_NOTE = ("Bohl, Lawrence, Lowry & Kaltenegger (2026), MNRAS, 'Probing the Limits of Habitability', "
             "Table 1 (Gaia DR3 + NEA). Zenodo 10.5281/zenodo.18134528, licence CC-BY-4.0.")

bohl_file = archive.latest_snapshot(RAW_DIR, "bohl2026_table1")
if bohl_file is None:
    print("Downloading Bohl et al. Table 1 from Zenodo ...")
    bohl_file = archive.download_file(BOHL_URL, RAW_DIR / f"bohl2026_table1_{date.today().isoformat()}.csv", BOHL_NOTE)
print("Loaded", bohl_file.relative_to(PROJECT_ROOT).as_posix())

# The file starts with '#' comment lines describing each column; the first column is a row number
bohl = pd.read_csv(bohl_file, comment="#", index_col=0)
bohl = bohl[bohl["pl_name"].notna()].reset_index(drop=True)
print(f"Bohl et al. Table 1: {len(bohl)} rocky planets in their empirical HZ, "
      f"stellar Teff {bohl['stellar_teff'].min():.0f}-{bohl['stellar_teff'].max():.0f} K")

in_range = hz.in_teff_range(bohl["stellar_teff"], K_RANGE)
compare = bohl.loc[in_range, ["pl_name", "stellar_teff", "innerFluxLimit", "3DinnerFluxLimit", "outerFluxLimit"]].copy()
compare["our_RV"] = hz.hz_flux_limit(compare["stellar_teff"], K13["recent_venus"], K_RANGE)
compare["our_3D"] = hz.hz_flux_limit(compare["stellar_teff"], hz.RAMIREZ_KALTENEGGER_2018_LECONTE, K_RANGE)
compare["our_EM"] = hz.hz_flux_limit(compare["stellar_teff"], K13["early_mars"], K_RANGE)
for ours, theirs in [("our_RV", "innerFluxLimit"), ("our_3D", "3DinnerFluxLimit"), ("our_EM", "outerFluxLimit")]:
    gap = (compare[ours] - compare[theirs]).abs()
    print(f"{ours} vs Bohl {theirs}: largest gap {gap.max():.3f} S⊕ over {len(compare)} planets")
compare.sort_values("stellar_teff").round(3).head(8)

### A second 3D zone: interpolating Bohl et al.'s own printed limits

Since we can't reproduce their 3D formula, we borrow their numbers directly. **Interpolation** means estimating a value between known points by drawing straight lines between them: if a 3000 K star has a limit of 0.80 and a 3100 K star 0.81, a 3050 K star gets 0.805. We interpolate their `3DinnerFluxLimit` and `outerFluxLimit` against stellar temperature. Stars outside their range (2566–5599 K) get "outside model range": we don't extrapolate. First check: using *their* fluxes, do we recover their 24 planets?

In [ ]:
# One limit per temperature (planets of the same star share it), sorted by temperature for np.interp
bohl_grid = bohl.groupby("stellar_teff")[["3DinnerFluxLimit", "outerFluxLimit"]].mean().sort_index()
BOHL_TEFF_RANGE = (bohl_grid.index.min(), bohl_grid.index.max())


def bohl_interpolated_limit(teff_values, column):
    """Straight-line interpolation of Bohl et al.'s printed limit; NaN outside their temperature range."""
    values = np.interp(teff_values, bohl_grid.index, bohl_grid[column])
    return np.where(hz.in_teff_range(teff_values, BOHL_TEFF_RANGE), values, np.nan)


def normalize_name(name):
    """Planet names for matching: lowercase, single spaces."""
    return " ".join(str(name).lower().split())


# Check 1: Bohl's own fluxes and temperatures
bohl_check = hz.hz_status(bohl["insolationFluxAvg"],
                          bohl_interpolated_limit(bohl["stellar_teff"], "3DinnerFluxLimit"),
                          bohl_interpolated_limit(bohl["stellar_teff"], "outerFluxLimit"),
                          bohl["stellar_teff"], BOHL_TEFF_RANGE)
bohl_3d_names = set(bohl.loc[(bohl["insolationFluxAvg"] <= bohl["3DinnerFluxLimit"])
                             & (bohl["insolationFluxAvg"] >= bohl["outerFluxLimit"]), "pl_name"].map(normalize_name))
print(f"Bohl et al.'s 3D HZ list: {len(bohl_3d_names)} planets")
print(f"Their fluxes + our interpolated limits -> inside: {(bohl_check == hz.INSIDE).sum()}")

# Our fluxes and temperatures
energy["bohl_3d_inner_interp"] = bohl_interpolated_limit(teff, "3DinnerFluxLimit")
energy["bohl_outer_interp"] = bohl_interpolated_limit(teff, "outerFluxLimit")
energy["hz_bohl_3d_interp"] = hz.hz_status(flux, energy["bohl_3d_inner_interp"], energy["bohl_outer_interp"],
                                           teff, BOHL_TEFF_RANGE)
our_3d_names = set(energy.loc[small & (energy["hz_bohl_3d_interp"] == hz.INSIDE), "pl_name"].map(normalize_name))
print(f"Our small planets inside the interpolated 3D zone: {len(our_3d_names)}")
print(f"  of Bohl's 24, we also have: {len(our_3d_names & bohl_3d_names)}")

In [ ]:
# Explain every disagreement with Bohl's 3D list
energy["name_key"] = energy["pl_name"].map(normalize_name)
bohl["name_key"] = bohl["pl_name"].map(normalize_name)
disagree = sorted((our_3d_names - bohl_3d_names) | (bohl_3d_names - our_3d_names))
explain = pd.DataFrame({"name_key": disagree})
explain["in"] = np.where(explain["name_key"].isin(bohl_3d_names), "Bohl only", "ours only")
explain = explain.merge(energy[["name_key", "st_teff", "hz_flux", "rocky_label", "hz_bohl_3d_interp",
                                 "new_since_bohl", "updated_since_bohl"]],
                        on="name_key", how="left")
explain = explain.merge(bohl[["name_key", "stellar_teff", "insolationFluxAvg", "3DinnerFluxLimit", "outerFluxLimit"]],
                        on="name_key", how="left")
explain.round(3)

### Step 5: the candidate table

Candidates are planets that are **rocky or possibly rocky** and **inside the Kopparapu (2014) optimistic zone** for their star's temperature.

**Flagged extras (your decision):** planets whose star is *outside* 2600–7200 K are not dropped silently. We evaluate the K14 limits at the **nearest valid temperature** (2600 K or 7200 K) instead of extrapolating the formula. Planets that land inside are added with a `model_range_caveat`. The strict `hz_k14_optimistic` flag still says "outside model range".

We sort by how close the flux is to Earth's (|S − 1|) and add the distance in **light years** (1 parsec = 3.2616 light years).

In [ ]:
energy["distance_ly"] = physics.parsecs_to_light_years(energy["sy_dist"])

# Limits at the nearest valid temperature, used only for the flagged extras
teff_clamped = hz.clamp_teff(teff, K_RANGE)
energy["hz_k14_optimistic_clamped"] = hz.hz_status(
    flux, hz.hz_flux_limit(teff_clamped, K14["recent_venus"]), hz.hz_flux_limit(teff_clamped, K14["early_mars"]),
    teff, None)
out_of_range = energy["hz_k14_optimistic"] == hz.OUTSIDE_MODEL_RANGE
flagged_extra = small & out_of_range & (energy["hz_k14_optimistic_clamped"] == hz.INSIDE)
energy["model_range_caveat"] = ""
energy.loc[flagged_extra, "model_range_caveat"] = (
    "limits evaluated at " + teff_clamped[flagged_extra].map("{:.0f} K".format)
    + " (star is " + teff[flagged_extra].map("{:.0f} K".format) + ")")

strict_candidate = small & (energy["hz_k14_optimistic"] == hz.INSIDE)
is_candidate = strict_candidate | flagged_extra
print(f"Rocky or possibly rocky:                          {small.sum():,}")
print(f"... inside the K14 optimistic zone (strict):      {strict_candidate.sum():,}")
print(f"... plus flagged out-of-range planets (clamped):  {flagged_extra.sum():,}  "
      f"{sorted(energy.loc[flagged_extra, 'pl_name'])}")
print(f"Candidates in the table:                          {is_candidate.sum():,}")
print(f"... of which inside K14 conservative:             {(is_candidate & (energy['hz_k14_conservative'] == hz.INSIDE)).sum():,}")

CANDIDATE_COLUMNS = [
    "pl_name", "hostname", "discoverymethod", "rocky_path", "rocky_label", "pl_rade", "pl_bmasse",
    "pl_bmassprov", "insol", "flux_orbit_avg", "hz_flux", "teq_albedo_0.3", "st_teff", "pl_orbper",
    "pl_orbeccen", "ecc_status", *HZ_COLUMNS, "hz_bohl_3d_interp", "hz_k14_optimistic_clamped",
    "model_range_caveat", "new_since_bohl", "updated_since_bohl", "sy_dist", "distance_ly",
]
candidates = energy.loc[is_candidate, CANDIDATE_COLUMNS].copy()
candidates["distance_from_earth_flux"] = (candidates["hz_flux"] - 1).abs()
candidates = candidates.sort_values("distance_from_earth_flux").reset_index(drop=True)

candidates.to_csv(PROCESSED_DIR / "hz_candidates.csv", index=False)
energy.to_csv(PROCESSED_DIR / "planets_classified.csv", index=False)
print("Saved data/processed/hz_candidates.csv", candidates.shape, "and planets_classified.csv", energy.shape)

candidates[["pl_name", "rocky_label", "pl_rade", "hz_flux", "teq_albedo_0.3", "st_teff",
            "hz_k14_conservative", "model_range_caveat", "distance_ly"]].head(15).round(2)

### What we learned in Phase 3

- **Our physics matches the archive.** Where the archive calculated S itself, our L/a² agrees to 0.00% (median). Where S comes from papers, the median gap is 6.8%. Our values filled the 46 gaps left by rounded-to-zero values.
- **Eccentricity hardly matters here.** Only 1,935 planets have a measured eccentricity (210 small ones), and switching every zone to the circular-orbit flux changes **0** planets.
- **The funnel:** 1,808 rocky or possibly rocky → 1,711 with a flux → 1,700 with a star inside 2600–7200 K → **45 in the Kopparapu (2014) optimistic zone**, plus **4 flagged TRAPPIST-1 planets** (d, e, f, g) whose 2566 K star is 34 K below the model range. That makes **49 candidates**, 28 of them in the conservative zone.
- **We reproduce Bohl et al.'s empirical limits** (Kopparapu 2013) to within 0.005 S⊕, but not their 3D limit. Interpolating their printed 3D limits instead recovers their 24 3D planets from their fluxes, and 22 of the 24 from ours.
- **Disagreements already have causes:** a planet discovered after their download (Gl 725 B c), a star temperature outside their table's range (Kepler-452 b), and flux differences of up to 4× (GJ 682 b) that come from different stellar and orbital parameters. Phase 6 digs into these.

## Phase 4: Visualize

Two figures: where every planet sits by **size and energy received**, and where the small planets sit relative to the **temperature-dependent habitable zone**.

**Reversed x axis.** We put high flux (hot) on the left and low flux (cold) on the right. Reading left to right then feels like moving outward from the star, which is how Kopparapu et al. draw their Fig. 3.

**Colormap and colorbar.** A colorbar is the legend for a color scale. Star temperature uses a *diverging* scale centered on the Sun (5772 K): red for stars cooler than the Sun, a light neutral for Sun-like stars, and blue for hotter ones. Temperatures beyond 2500–7500 K get the end colors (the arrow tips on the colorbar).

**The shaded zones in the first plot are only for a Sun-like star.** Habitable zone edges depend on each star's temperature, so a single radius–flux plot can only shade the Sun's edges. The second plot shows the real, temperature-dependent curves.

In [ ]:
from src import plot_labels

plot_style.apply_style()
teff_cmap, teff_norm = plot_style.teff_colormap()

candidates_plot = energy[is_candidate].copy()
candidates_plot["closeness"] = (candidates_plot["hz_flux"] - 1).abs()
candidates_plot = candidates_plot.sort_values("closeness")
TOP_N = 12
top = candidates_plot.head(TOP_N)
print(f"Labelling the {TOP_N} candidates closest to Earth's flux:")
print(", ".join(top["pl_name"]))

### Plot 1: planet radius vs energy received

Every planet with a **measured** radius is a dot colored by its star's temperature. Habitable zone candidates get a dark ring. Candidates on the *mass only* path have no measured radius, so they are drawn **hollow** at the radius the archive estimated from their mass. Otherwise half the candidates, including the two nearest, would be missing. The dashed line is our rocky cutoff (1.6 R⊕) and the dotted line the possibly-rocky limit (2.0 R⊕).

In [ ]:
X_RANGE = (1e4, 1e-2)   # reversed: hot on the left
Y_RANGE = (0.3, 30)

measured = energy[(energy["rocky_path"] == rocky.PATH_MEASURED_RADIUS) & energy["hz_flux"].notna()]
in_window = (measured["hz_flux"].between(X_RANGE[1], X_RANGE[0])
             & measured["pl_rade"].between(*Y_RANGE))
print(f"Measured-radius planets with a flux: {len(measured):,}; "
      f"outside the plotted window: {(~in_window).sum()} (extreme flux or radius)")

fig, ax = plt.subplots(figsize=(12, 8))

# Sun-like habitable zone bands (Kopparapu et al. 2014, Table 1 values for the Sun)
ax.axvspan(K14["early_mars"][0], K14["recent_venus"][0], color=plot_style.SHADE, zorder=0,
           label="Optimistic HZ, 0.32-1.78 S⊕ (Sun-like star)")
ax.axvspan(K14["maximum_greenhouse"][0], K14["runaway_greenhouse"][0], color=plot_style.SHADE_DARK, zorder=0,
           label="Conservative HZ, 0.36-1.11 S⊕ (Sun-like star)")
ax.axhline(rocky.ROCKY_MAX_RADIUS, color=plot_style.TEXT_SECONDARY, linestyle="--", linewidth=1,
           label="Rocky cutoff, 1.6 R⊕ (Rogers 2015)")
ax.axhline(rocky.POSSIBLY_ROCKY_MAX_RADIUS, color=plot_style.TEXT_SECONDARY, linestyle=":", linewidth=1,
           label="Possibly-rocky limit, 2.0 R⊕ (Bohl et al. 2026)")

points = ax.scatter(measured["hz_flux"], measured["pl_rade"], c=measured["st_teff"], cmap=teff_cmap,
                    norm=teff_norm, s=14, alpha=0.8, edgecolors=plot_style.TEXT_MUTED, linewidths=0.25,
                    zorder=2, label="Planet with a measured radius")

cand_measured = candidates_plot[candidates_plot["rocky_path"] == rocky.PATH_MEASURED_RADIUS]
cand_mass_only = candidates_plot[candidates_plot["rocky_path"] == rocky.PATH_MASS_ONLY]
ax.scatter(cand_measured["hz_flux"], cand_measured["pl_rade"], c=cand_measured["st_teff"], cmap=teff_cmap,
           norm=teff_norm, s=70, edgecolors=plot_style.RING, linewidths=1.4, zorder=4,
           label=f"HZ candidate, measured radius ({len(cand_measured)})")
ax.scatter(cand_mass_only["hz_flux"], cand_mass_only["pl_rade"], facecolors="none",
           edgecolors=plot_style.RING, s=70, linewidths=1.4, zorder=4,
           label=f"HZ candidate, radius estimated from mass, not measured ({len(cand_mass_only)})")

# Earth: a circle with a cross, the astronomical symbol for Earth
ax.scatter([1], [1], s=150, facecolors=plot_style.SURFACE, edgecolors=plot_style.SERIES[2], linewidths=2, zorder=5)
ax.scatter([1], [1], s=150, marker="+", color=plot_style.SERIES[2], linewidths=2, zorder=5, label="Earth")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(*X_RANGE)
ax.set_ylim(*Y_RANGE)
ax.set_yticks([0.5, 1, 1.6, 2, 3, 5, 10, 20])
ax.set_yticklabels(["0.5", "1", "1.6", "2", "3", "5", "10", "20"])
ax.set_xlabel("Energy received, orbit-averaged where measured (Earth = 1, log scale, hotter to the left)")
ax.set_ylabel("Planet radius (Earth radii, log scale)")
ax.set_title("Exoplanets by size and energy received, with habitable zone candidates")
colorbar = fig.colorbar(points, ax=ax, extend="both", pad=0.015)
colorbar.set_label("Host star temperature (K)")
colorbar.ax.axhline(plot_style.SUN_TEFF, color=plot_style.TEXT_PRIMARY, linewidth=1)
colorbar.ax.text(1.6, plot_style.SUN_TEFF, " Sun", transform=colorbar.ax.get_yaxis_transform(),
                 va="center", fontsize=8, color=plot_style.TEXT_SECONDARY)
fig.legend(loc="lower center", ncol=2, bbox_to_anchor=(0.45, -0.08), fontsize=9)

labels_1, skipped_1 = plot_labels.annotate_without_overlap(
    ax, top["hz_flux"], top["pl_rade"], top["pl_name"],
    obstacle_xs=list(candidates_plot["hz_flux"]) + [1.0], obstacle_ys=list(candidates_plot["pl_rade"]) + [1.0])
assert not plot_labels.text_boxes_overlap(labels_1), "labels overlap"
print(f"Labels drawn: {len(labels_1)}, could not place: {skipped_1}, overlaps: none")

fig.savefig(FIGURES_DIR / "radius_vs_insolation.png")
plt.show()

### Plot 2: the temperature-dependent habitable zone

Here every small (rocky or possibly rocky) planet is placed by the flux it receives and its star's temperature. The curves are the Kopparapu et al. (2014) edges: **Recent Venus** and **Early Mars** bound the optimistic zone (light band), and **Runaway Greenhouse** and **Maximum Greenhouse** bound the conservative zone (darker band). The formulas are valid only for 2600–7200 K, so the curves stop there, and the hatched bands mark stars outside the model's range. The left panel shows everything; the right panel zooms in on the habitable zone.

In [ ]:
import matplotlib.ticker

small_plot = energy[small & energy["hz_flux"].notna() & energy["st_teff"].notna()]
Y_TEFF = (2400, 7600)
X_ALL = (2e4, 0.05)
X_ZOOM = (3, 0.1)
beyond = ~small_plot["st_teff"].between(*Y_TEFF) | ~small_plot["hz_flux"].between(X_ALL[1], X_ALL[0])
print(f"Small planets with flux and Teff: {len(small_plot):,}; outside the left panel: {beyond.sum()} "
      f"{sorted(small_plot.loc[beyond, 'pl_name'])}")

curve_teff = np.linspace(*K_RANGE, 200)
curves = {name: hz.hz_flux_limit(curve_teff, K14[name]) for name in K14}

fig, (ax_all, ax_zoom) = plt.subplots(1, 2, figsize=(15, 7), sharey=True, gridspec_kw={"width_ratios": [3, 2]})


def draw_hz(ax):
    """Kopparapu (2014) bands and curves, plus hatching outside the model's temperature range."""
    ax.fill_betweenx(curve_teff, curves["early_mars"], curves["recent_venus"], color=plot_style.SHADE, zorder=0,
                     label="Optimistic HZ (Recent Venus to Early Mars)")
    ax.fill_betweenx(curve_teff, curves["maximum_greenhouse"], curves["runaway_greenhouse"],
                     color=plot_style.SHADE_DARK, zorder=0,
                     label="Conservative HZ (Runaway to Maximum Greenhouse)")
    for name, style in [("recent_venus", "-"), ("early_mars", "-"),
                        ("runaway_greenhouse", "--"), ("maximum_greenhouse", "--")]:
        ax.plot(curves[name], curve_teff, color=plot_style.TEXT_SECONDARY, linestyle=style, linewidth=1, zorder=1)
    for low, high in [(Y_TEFF[0], K_RANGE[0]), (K_RANGE[1], Y_TEFF[1])]:
        ax.axhspan(low, high, facecolor="none", edgecolor=plot_style.GRID, hatch="///", linewidth=0, zorder=0)


def draw_planets(ax):
    """Small planets colored by rocky label; hollow = mass only; ringed = candidate."""
    for label, color in [("rocky", plot_style.SERIES[0]), ("possibly rocky", plot_style.SERIES[1])]:
        for path, filled in [(rocky.PATH_MEASURED_RADIUS, True), (rocky.PATH_MASS_ONLY, False)]:
            group = small_plot[(small_plot["rocky_label"] == label) & (small_plot["rocky_path"] == path)]
            if group.empty:
                continue  # e.g. mass-only planets are never "rocky"
            name = f"{label.capitalize()}, {'measured radius' if filled else 'mass only'} ({len(group)})"
            ax.scatter(group["hz_flux"], group["st_teff"], s=16, linewidths=0.9, zorder=2, alpha=0.85,
                       facecolors=color if filled else "none", edgecolors=color, label=name)
    ring = small_plot[small_plot["pl_name"].isin(candidates_plot["pl_name"])]
    ax.scatter(ring["hz_flux"], ring["st_teff"], s=60, facecolors="none", edgecolors=plot_style.RING,
               linewidths=1.2, zorder=3, label=f"HZ candidate ({len(ring)})")
    ax.scatter([1.0], [plot_style.SUN_TEFF], s=150, marker="+", color=plot_style.SERIES[2], linewidths=2, zorder=5,
               label="Earth, Venus (1.91 S⊕), Mars (0.43 S⊕)")
    ax.scatter([1.0, 1.91, 0.43], [plot_style.SUN_TEFF] * 3, s=150, facecolors="none",
               edgecolors=plot_style.SERIES[2], linewidths=2, zorder=5)


for ax, x_range, title in [(ax_all, X_ALL, "All small planets"), (ax_zoom, X_ZOOM, "Zoom on the habitable zone")]:
    draw_hz(ax)
    draw_planets(ax)
    ax.set_xscale("log")
    ax.set_xlim(*x_range)
    ax.set_title(title)
    ax.set_xlabel("Energy received (Earth = 1, log scale, hotter to the left)")
ax_all.set_ylim(*Y_TEFF)
ax_all.set_ylabel("Host star temperature (K)")
ax_all.text(X_ALL[0] * 0.8, 2500, "outside model range (< 2600 K)", fontsize=8, color=plot_style.TEXT_SECONDARY, va="center")
ax_all.text(X_ALL[0] * 0.8, 7400, "outside model range (> 7200 K)", fontsize=8, color=plot_style.TEXT_SECONDARY, va="center")
ax_zoom.set_xticks([3, 2, 1, 0.5, 0.3, 0.2, 0.1])
ax_zoom.set_xticklabels(["3", "2", "1", "0.5", "0.3", "0.2", "0.1"])
ax_zoom.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())

fig.suptitle("Small planets and the temperature-dependent habitable zone (Kopparapu et al. 2014)", fontsize=14,
             color=plot_style.TEXT_PRIMARY)
handles, names = ax_all.get_legend_handles_labels()
fig.legend(handles, names, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.1), fontsize=9)
fig.tight_layout()

zoom_points = small_plot[small_plot["hz_flux"].between(X_ZOOM[1], X_ZOOM[0])]
labels_2, skipped_2 = plot_labels.annotate_without_overlap(
    ax_zoom, [1.91, 1.0, 0.43] + list(top["hz_flux"]), [plot_style.SUN_TEFF] * 3 + list(top["st_teff"]),
    ["Venus", "Earth", "Mars"] + list(top["pl_name"]),
    obstacle_xs=list(zoom_points["hz_flux"]) + [1.91, 1.0, 0.43],
    obstacle_ys=list(zoom_points["st_teff"]) + [plot_style.SUN_TEFF] * 3,
    obstacle_radius=4)
assert not plot_labels.text_boxes_overlap(labels_2), "labels overlap"
print(f"Labels drawn: {len(labels_2)}, could not place: {skipped_2}, overlaps: none")

fig.savefig(FIGURES_DIR / "teff_vs_insolation_hz.png")
plt.show()

### What we learned in Phase 4

- **Almost every candidate orbits a cool red star.** 41 of the 49 candidates orbit M dwarfs (below 4000 K), 7 orbit K stars, and only Kepler-452 b orbits a Sun-like star. In the first plot, the ringed candidates are nearly all red.
- **Candidates mostly get less energy than Earth** (31 of 49; median 0.83 S⊕). The second plot shows why: for cooler stars the habitable zone curves bend toward lower flux, because red light warms a planet more efficiently.
- **Two kinds of candidates, two kinds of distance.** The 21 mass-only candidates (hollow) are nearby radial velocity planets (median 18 light years), while measured-radius candidates are mostly distant Kepler and TESS transit planets (median 423 light years). 20 of the 25 candidates within 50 light years have no measured radius.
- **Detection bias is visible.** The top-left of the first plot is crowded with hot giant planets, which are the easiest to find. Small planets that receive little energy (bottom right) are rare because they are hard to detect, not necessarily because they are rare in nature.
- **TRAPPIST-1 sits in the hatched band** just below the 2600 K model limit, which shows on the figure why those four candidates carry a caveat.

## Phase 5: Clustering

Can a computer, given only a few numbers per planet, discover natural groups of planets on its own? And where would Earth fit? This is **unsupervised** learning: nobody tells the algorithm what the right groups are.

**Which planets.** Only planets with a **measured** radius. The archive's estimated radii create an artificial spike at 12–14 R⊕ (Phase 2) that KMeans could mistake for a real group.

**Features:** log radius, log flux (`hz_flux`), log orbital period, and stellar temperature.
- **Why logs:** radius, flux and period span many powers of ten (period: 0.09 days to centuries). Without logs, the few largest values would dominate every distance. With logs, "twice as big" is the same step everywhere.
- **Why StandardScaler:** KMeans measures plain straight-line distance. Teff in thousands of kelvin would swamp log values between about −2 and 4, so each feature is rescaled to mean 0 and standard deviation 1 ("how many standard deviations from average"). Then all four features count equally.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from src import clustering

measured_path = energy["rocky_path"] == rocky.PATH_MEASURED_RADIUS
features = clustering.build_features(energy[measured_path])
print(f"Planets in the table:              {len(energy):,}")
print(f"... with a measured radius:        {measured_path.sum():,}")
print(f"... and all four features:         {len(features):,}  (these are clustered)")
missing = energy.loc[measured_path, ["hz_flux", "pl_orbper", "st_teff"]].isna().sum()
print("    missing among measured-radius planets:", missing.to_dict())

hot_stars = energy.loc[features.index][features["st_teff"] > 8000]
print(f"\nStars hotter than 8000 K kept in (outliers, not removed): {len(hot_stars)}, "
      f"hottest {hot_stars['st_teff'].max():.0f} K ({', '.join(hot_stars.nlargest(2, 'st_teff')['pl_name'])})")

scaler = StandardScaler().fit(features)
X = scaler.transform(features)
earth_X = scaler.transform(clustering.earth_features())
print("\nScaler means:", dict(zip(clustering.FEATURE_COLUMNS, scaler.mean_.round(2))))
print("Scaler standard deviations:", dict(zip(clustering.FEATURE_COLUMNS, scaler.scale_.round(2))))

### Choosing the number of clusters k

**KMeans** places k centers, assigns every planet to its nearest center, moves each center to the average of its members, and repeats until nothing changes. The result depends on where the centers start, so `n_init=10` runs it 10 times and keeps the best, and `random_state` fixes the random starts so the result is reproducible.

We judge each k three ways:
- **Elbow (inertia):** the total squared distance from planets to their centers. It always drops as k grows; we look for where the drop slows down (the "elbow").
- **Silhouette score:** for each planet, how much closer it is to its own cluster than to the next nearest one, averaged over all planets. It runs from −1 to 1; higher means cleaner, better-separated clusters.
- **Stability (adjusted Rand index):** we rerun with seeds 0–4 and compare each run with seed 0. The adjusted Rand index is 1 when two clusterings group the planets identically (whatever numbers the clusters get) and about 0 when they agree only by chance. We report the worst of the four comparisons.

In [ ]:
SEEDS = (0, 1, 2, 3, 4)
rows = []
for k in range(2, 11):
    model = clustering.fit_kmeans(X, k, seed=0)
    rows.append({"k": k, "inertia": model.inertia_, "silhouette": silhouette_score(X, model.labels_),
                 "worst_seed_agreement": min(clustering.seed_stability(X, k, SEEDS)),
                 "smallest_cluster": np.bincount(model.labels_).min()})
k_table = pd.DataFrame(rows).set_index("k")
k_table.round(3)

In [ ]:
CHOSEN_K = 4

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
panels = [("inertia", "Elbow: total squared distance", "Inertia (scaled units²)"),
          ("silhouette", "Silhouette score (higher is better)", "Mean silhouette"),
          ("worst_seed_agreement", "Stability across seeds 0-4", "Worst adjusted Rand index")]
for ax, (column, title, ylabel) in zip(axes, panels):
    ax.plot(k_table.index, k_table[column], color=plot_style.SERIES[0], linewidth=2, marker="o", markersize=6)
    ax.axvline(CHOSEN_K, color=plot_style.TEXT_SECONDARY, linestyle="--", linewidth=1, label=f"Chosen k = {CHOSEN_K}")
    ax.set_title(title)
    ax.set_xlabel("Number of clusters k")
    ax.set_ylabel(ylabel)
    ax.set_xticks(k_table.index)
axes[0].legend(loc="upper right", fontsize=9)  # one legend: the dashed line means the same in every panel
fig.suptitle("Choosing k for KMeans (4,622 planets with measured radii)", fontsize=13, color=plot_style.TEXT_PRIMARY)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "kmeans_choosing_k.png")
plt.show()

**Choosing k = 4.** Silhouette peaks at k = 4 and k = 5 (0.353 vs 0.354, a tie). k = 5 fails the other two tests: its extra cluster contains just **2 planets** (KOI-55 b and c, around a 27,730 K star), and it is unstable, with one seed agreeing with seed 0 at only about 0.54. k = 4 is stable (worst agreement about 0.98) and every cluster is large. The elbow bends around k = 4 as well. Silhouette does **not** favor k = 2 here, but we still look at a larger k (k = 6) below, because "best separated" and "most informative" are not the same thing.

In [ ]:
model_k4 = clustering.fit_kmeans(X, CHOSEN_K, seed=0)
# Renumber clusters from hottest (highest median flux) to coolest so numbers and colors are stable
cluster_k4, k4_mapping = clustering.order_clusters(model_k4.labels_, features["log_flux"].to_numpy())

clustered = energy.loc[features.index, ["pl_name", "hostname", "pl_rade", "hz_flux", "pl_orbper", "st_teff",
                                       "rocky_label", "discoverymethod"]].copy()
clustered["cluster"] = cluster_k4


def describe_clusters(table, column):
    summary = table.groupby(column).agg(
        planets=("pl_name", "size"), radius=("pl_rade", "median"), flux=("hz_flux", "median"),
        period_days=("pl_orbper", "median"), star_teff=("st_teff", "median"))
    return summary.round({"radius": 2, "flux": 1, "period_days": 1, "star_teff": 0})


k4_summary = describe_clusters(clustered, "cluster")
k4_summary

**Naming the clusters from their medians** (median radius, flux, period, star temperature):
- **0, hot giant planets:** about 13 R⊕ (Jupiter size), hundreds of times Earth's flux, orbits of a few days. These are hot Jupiters.
- **1, hot small planets of Sun-like stars:** under 2 R⊕, over 200× Earth's flux, orbits of about 6 days around stars near 5700 K.
- **2, planets of cool stars:** small planets around K and M dwarfs (median star about 3900 K). Their flux is much lower even though their orbits are just as short.
- **3, warm sub-Neptunes of Sun-like stars:** about 2.8 R⊕, roughly 18× Earth's flux, orbits of about 40 days.

The code below checks these names against the medians, so a change in the data can't silently mislabel a cluster.

In [ ]:
CLUSTER_NAMES = {0: "Hot giant planets", 1: "Hot small planets, Sun-like stars",
                 2: "Planets of cool stars", 3: "Warm sub-Neptunes, Sun-like stars"}
assert k4_summary.loc[0, "radius"] > 8                                   # giants
assert k4_summary.loc[1, "radius"] < 2.2 and k4_summary.loc[1, "flux"] > 100
assert k4_summary.loc[2, "star_teff"] < 4500                             # cool stars
assert k4_summary.loc[3, "period_days"] > 20 and k4_summary.loc[3, "star_teff"] > 5000
clustered["cluster_name"] = clustered["cluster"].map(CLUSTER_NAMES)
k4_summary.assign(name=k4_summary.index.map(CLUSTER_NAMES))

### Where does Earth fit?

We pass Earth (1 R⊕, 1 S⊕, a 365.256-day orbit, a 5772 K star) through the **same** scaler and model. KMeans will always assign it to *some* cluster, so we also ask how far Earth is from that cluster's center compared with the cluster's real members.

In [ ]:
earth_raw_cluster = model_k4.predict(earth_X)[0]
earth_cluster = k4_mapping[earth_raw_cluster]
center = model_k4.cluster_centers_[earth_raw_cluster]
members = clustered["cluster"] == earth_cluster
member_distance = np.linalg.norm(X[members.to_numpy()] - center, axis=1)
earth_distance = np.linalg.norm(earth_X[0] - center)

print(f"Earth is assigned to cluster {earth_cluster}: {CLUSTER_NAMES[earth_cluster]} ({members.sum():,} planets)")
print(f"Earth's distance to the cluster center: {earth_distance:.2f} (scaled units)")
print(f"Typical member's distance (median):     {np.median(member_distance):.2f}; "
      f"90th percentile: {np.percentile(member_distance, 90):.2f}")
print(f"Earth is farther from the center than {(member_distance < earth_distance).mean() * 100:.1f}% of members.")

earth_offsets = pd.Series(earth_X[0] - center, index=clustering.FEATURE_COLUMNS).round(2)
print("\nEarth minus the cluster center, per feature (in standard deviations):")
print(earth_offsets.to_string())

clustered["distance_to_earth_4feat"] = np.linalg.norm(X - earth_X[0], axis=1)
in_hz = clustered["pl_name"].isin(candidates_plot["pl_name"])
print(f"\nHZ candidates in Earth's cluster: {(members & in_hz).sum()} of {in_hz.sum()} candidates that have measured radii")
print("Members of Earth's cluster closest to Earth:")
clustered[members].nsmallest(10, "distance_to_earth_4feat")[
    ["pl_name", "pl_rade", "hz_flux", "pl_orbper", "st_teff", "distance_to_earth_4feat"]].round(2)

### What does a larger k reveal? (k = 6)

k = 6 scores slightly lower on silhouette (0.326) but is just as stable. The question is whether the extra clusters are physically meaningful.

In [ ]:
model_k6 = clustering.fit_kmeans(X, 6, seed=0)
cluster_k6, k6_mapping = clustering.order_clusters(model_k6.labels_, features["log_flux"].to_numpy())
clustered["cluster_k6"] = cluster_k6
earth_cluster_k6 = k6_mapping[model_k6.predict(earth_X)[0]]
center_k6 = model_k6.cluster_centers_[model_k6.predict(earth_X)[0]]
members_k6 = clustered["cluster_k6"].to_numpy() == earth_cluster_k6
member_distance_k6 = np.linalg.norm(X[members_k6] - center_k6, axis=1)
earth_distance_k6 = np.linalg.norm(earth_X[0] - center_k6)
print(f"Earth's cluster at k = 6: {earth_cluster_k6} ({members_k6.sum():,} planets); Earth is farther from its center "
      f"({earth_distance_k6:.2f}) than {(member_distance_k6 < earth_distance_k6).mean() * 100:.1f}% of members "
      f"(median {np.median(member_distance_k6):.2f})")
print(f"Agreement between k = 4 and k = 6 groupings (adjusted Rand): "
      f"{adjusted_rand_score(clustered['cluster'], clustered['cluster_k6']):.2f}")
print("How the k = 4 clusters split up at k = 6 (rows: k = 4, columns: k = 6):")
print(pd.crosstab(clustered["cluster"], clustered["cluster_k6"]).to_string())
describe_clusters(clustered, "cluster_k6")

**Which k is more useful scientifically?** k = 4 gives the robust big picture: hot giants, hot small planets, planets of cool stars, and warm sub-Neptunes. At k = 6 (and every k ≥ 5) one cluster is just KOI-55 b and c, two planets around a 27,730 K hot subdwarf star. That is a real oddity, but not a population. The other new split divides the planets of Sun-like stars by **how much energy they get and how long their orbits are**: hot rocky planets (about 1.6 R⊕, more than 400× Earth's flux, 4-day orbits), warm sub-Neptunes (about 2.5 R⊕, 60×, 16 days), and a cooler, longer-period group (about 3.2 R⊕, 6×, 87 days) that Earth joins. That last group is a better set of neighbors for "planets in an Earth-like situation", so k = 6 is more useful for finding Earth analogues, while k = 4 is the better summary of the whole population (higher silhouette, and an agreement of only 0.54 with k = 6 shows the finer split is a real reorganization, not just extra detail). We keep k = 4 as the main `cluster` column and k = 6 as `cluster_k6` for Phase 6.

### Seeing the clusters

First, the clusters on the same radius vs flux axes as Phase 4. Then a **PCA** view. PCA (principal component analysis) finds the direction through the cloud of points where the planets are most spread out, then the next most spread-out direction at right angles to the first, and draws every planet on those two directions. It is like photographing a 3D object from the angle that shows the most detail. The arrows show which way each original feature increases in that picture.

In [ ]:
CLUSTER_STYLE = {  # three hues for the small-planet clusters; dark neutral for the giants
    0: (plot_style.TEXT_SECONDARY, "D"), 1: (plot_style.SERIES[0], "o"),
    2: (plot_style.SERIES[1], "s"), 3: (plot_style.SERIES[2], "^"),
}

fig, ax = plt.subplots(figsize=(12, 7.5))
for number, (color, marker) in CLUSTER_STYLE.items():
    group = clustered[clustered["cluster"] == number]
    ax.scatter(group["hz_flux"], group["pl_rade"], s=14, color=color, marker=marker, alpha=0.6, linewidths=0,
               label=f"{number}: {CLUSTER_NAMES[number]} ({len(group):,})")
ax.scatter([1], [1], s=220, marker="*", color=plot_style.TEXT_PRIMARY, edgecolors=plot_style.SURFACE,
           linewidths=1, zorder=5, label=f"Earth (assigned to cluster {earth_cluster})")
ax.axhline(rocky.ROCKY_MAX_RADIUS, color=plot_style.TEXT_SECONDARY, linestyle="--", linewidth=1,
           label="Rocky cutoff, 1.6 R⊕")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1e4, 1e-2)
ax.set_ylim(0.3, 30)
ax.set_yticks([0.5, 1, 1.6, 2, 3, 5, 10, 20])
ax.set_yticklabels(["0.5", "1", "1.6", "2", "3", "5", "10", "20"])
ax.set_xlabel("Energy received (Earth = 1, log scale, hotter to the left)")
ax.set_ylabel("Planet radius (Earth radii, log scale)")
ax.set_title(f"KMeans clusters (k = {CHOSEN_K}) on radius vs energy received")
fig.legend(loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.07), fontsize=9, markerscale=1.5)

# Direct labels at each cluster's median position
medians = clustered.groupby("cluster")[["hz_flux", "pl_rade"]].median()
cluster_labels, _ = plot_labels.annotate_without_overlap(
    ax, medians["hz_flux"], medians["pl_rade"], [CLUSTER_NAMES[n] for n in medians.index], fontsize=9)
assert not plot_labels.text_boxes_overlap(cluster_labels)
fig.savefig(FIGURES_DIR / "clusters_radius_vs_insolation.png")
plt.show()

In [ ]:
pca = PCA(n_components=2).fit(X)
pcs = pca.transform(X)
earth_pc = pca.transform(earth_X)[0]
explained = pca.explained_variance_ratio_ * 100
print(f"PC1 explains {explained[0]:.0f}% of the spread, PC2 {explained[1]:.0f}%, together {explained.sum():.0f}%")

fig, ax = plt.subplots(figsize=(10, 7.5))
for number, (color, marker) in CLUSTER_STYLE.items():
    mask = clustered["cluster"].to_numpy() == number
    ax.scatter(pcs[mask, 0], pcs[mask, 1], s=12, color=color, marker=marker, alpha=0.55, linewidths=0,
               label=f"{number}: {CLUSTER_NAMES[number]}")
ax.scatter(*earth_pc, s=260, marker="*", color=plot_style.TEXT_PRIMARY, edgecolors=plot_style.SURFACE,
           linewidths=1, zorder=5, label="Earth")

# Arrows: the direction in which each original feature increases
FEATURE_LABELS = {"log_radius": "bigger", "log_flux": "more flux", "log_period": "longer orbit",
                  "st_teff": "hotter star"}
arrow_scale = 2.5
tips = []
for name, (dx, dy) in zip(clustering.FEATURE_COLUMNS, pca.components_.T):
    ax.annotate("", xy=(dx * arrow_scale, dy * arrow_scale), xytext=(0, 0), zorder=6,
                arrowprops={"arrowstyle": "->", "color": plot_style.TEXT_PRIMARY, "linewidth": 1.4})
    tips.append((dx * arrow_scale, dy * arrow_scale, FEATURE_LABELS[name]))
ax.set_xlabel(f"Principal component 1 ({explained[0]:.0f}% of the spread)")
ax.set_ylabel(f"Principal component 2 ({explained[1]:.0f}% of the spread)")
ax.set_title("Clusters in a 2D PCA view of the four scaled features")
PC_LIMITS = (-6, 6), (-4.5, 4.5)  # zoom on the main cloud; a few outliers fall outside (counted below)
ax.set_xlim(*PC_LIMITS[0])
ax.set_ylim(*PC_LIMITS[1])
outside = ~((pcs[:, 0] >= PC_LIMITS[0][0]) & (pcs[:, 0] <= PC_LIMITS[0][1])
            & (pcs[:, 1] >= PC_LIMITS[1][0]) & (pcs[:, 1] <= PC_LIMITS[1][1]))
print(f"Planets outside the plotted PCA window: {outside.sum()} "
      f"({', '.join(clustered.loc[outside, 'pl_name'].head(6))}{'...' if outside.sum() > 6 else ''})")
fig.legend(loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.08), fontsize=9, markerscale=1.5)
pca_labels, _ = plot_labels.annotate_without_overlap(
    ax, [t[0] for t in tips] + [earth_pc[0]], [t[1] for t in tips] + [earth_pc[1]],
    [t[2] for t in tips] + ["Earth"], fontsize=9)
assert not plot_labels.text_boxes_overlap(pca_labels)
fig.savefig(FIGURES_DIR / "clusters_pca.png")
plt.show()

pd.DataFrame(pca.components_.T, index=clustering.FEATURE_COLUMNS, columns=["PC1", "PC2"]).round(2)

### Two similarity-to-Earth scores

**1. Four-feature similarity:** distance from Earth in the scaled space of all four features (size, energy, orbital period, star temperature), turned into a score: **similarity = 1 / (1 + distance)**. Earth scores exactly 1.

**2. Earth Similarity Index (ESI), simplified version for exoplanets:** it uses only radius R and flux S, both in Earth units:

ESI = 1 − √( ½ [ ((S − 1)/(S + 1))² + ((R − 1)/(R + 1))² ] )

Each term measures how different a planet is from Earth as a fraction, so a planet twice Earth's size counts the same whether Earth is the bigger or smaller one in the pair. Earth scores exactly 1. Source: Dabhade et al. (2025, arXiv:2512.00899), eq. 7, following Schulze-Makuch et al. (2011, Astrobiology 11, 1041). Pratyush & Gangrade (2021, arXiv:2112.03298) describe the same two-parameter ESI. We use `hz_flux` for S.

The four-feature score rewards planets that also have a year-long orbit around a Sun-like star. The ESI ignores the star, so planets of small red stars can rank highly.

In [ ]:
clustered["similarity_4feat"] = clustering.similarity_to_earth(X, earth_X)
clustered["esi"] = clustering.earth_similarity_index(clustered["hz_flux"], clustered["pl_rade"])
clustered["rank_4feat"] = clustered["similarity_4feat"].rank(ascending=False, method="min").astype(int)
clustered["rank_esi"] = clustered["esi"].rank(ascending=False, method="min").astype(int)
clustered["hz_candidate"] = in_hz

show = ["pl_name", "pl_rade", "hz_flux", "pl_orbper", "st_teff", "hz_candidate"]
top_4 = clustered.nsmallest(10, "rank_4feat")[show + ["similarity_4feat"]].reset_index(drop=True)
top_esi = clustered.nsmallest(10, "rank_esi")[show + ["esi"]].reset_index(drop=True)
both = set(top_4["pl_name"]) & set(top_esi["pl_name"])
print(f"Planets in both top-10 lists: {len(both)} {sorted(both)}")
print(f"HZ candidates in the 4-feature top 10: {top_4['hz_candidate'].sum()}, in the ESI top 10: {top_esi['hz_candidate'].sum()}")
print(f"Median star temperature, 4-feature top 10: {top_4['st_teff'].median():.0f} K; "
      f"ESI top 10: {top_esi['st_teff'].median():.0f} K")
print("\nTop 10 by all four features:")
print(top_4.round(3).to_string())
print("\nTop 10 by Earth Similarity Index (radius and flux):")
print(top_esi.round(3).to_string())

**What changed compared with our earlier unweighted radius+flux score:** 9 of the 10 planets are the same. Gliese 12 b dropped out and Kepler-452 b came in, because the ESI's fractional differences treat Kepler-452 b's 1.63 R⊕ more gently than distance in scaled log space did.

**Caveat, Kepler-1649 c:** its rank depends on which flux is right. With our flux (0.75 S⊕, Vanderburg et al. 2020) it ranks 4th. With Bohl et al.'s 1.12 S⊕ its ESI would be about 0.955, which would rank it **first**. Phase 6 checks which flux is more trustworthy.

In [ ]:
k1649 = clustered["pl_name"] == "Kepler-1649 c"
radius_1649 = clustered.loc[k1649, "pl_rade"].iloc[0]
esi_with_bohl_flux = clustering.earth_similarity_index(1.12, radius_1649)
print(f"Kepler-1649 c: ESI {clustered.loc[k1649, 'esi'].iloc[0]:.3f} (rank {clustered.loc[k1649, 'rank_esi'].iloc[0]}) "
      f"with our flux; {esi_with_bohl_flux:.3f} with Bohl's 1.12 S⊕, which would be rank "
      f"{(clustered.loc[~k1649, 'esi'] > esi_with_bohl_flux).sum() + 1}")

In [ ]:
clustered_file = PROCESSED_DIR / "planets_clustered.csv"
clustered.join(features.drop(columns="st_teff")).to_csv(clustered_file, index=False)
print("Saved", clustered_file.relative_to(PROJECT_ROOT).as_posix(), clustered.shape)

### What we learned in Phase 5

- **KMeans finds four robust groups among 4,622 planets with measured radii:** hot giants (813), hot small planets of Sun-like stars (1,888), planets of cool stars (775), and warm sub-Neptunes of Sun-like stars (1,146). k = 4 has the best silhouette (tied with k = 5) and agrees with itself across seeds (worst adjusted Rand index 0.98). Every k ≥ 5 spends a cluster on the 2 KOI-55 planets, and k = 5 is unstable.
- **Earth is an outlier.** It is assigned to the warm sub-Neptunes but lies farther from the center than 96.8% of members (91% at k = 6). Compared with the center, Earth is about 1.4 standard deviations smaller, 1.2 cooler in flux, and 1.5 longer in orbit. That is detection bias: small planets on year-long orbits around Sun-like stars are the hardest to find.
- **The two similarity scores mostly disagree.** With all four features, Kepler planets around Sun-like stars lead (Kepler-452 b first; median star 5686 K). With the Earth Similarity Index (radius and flux), planets of small red stars lead (TOI-700 d, TOI-700 e, TRAPPIST-1 d; 9 of 10 are our HZ candidates). Kepler-452 b is the only planet in both top 10s.
- **What "Earth-like" means depends on the question.** If the star and the orbit matter (similar seasons, similar light), Earth's analogues are rare Kepler planets. If only size and energy matter (ESI), the best analogues orbit M dwarfs. That is also where the habitable zone candidates are.

## Phase 6: Compare with NASA's own research